# Cleanse, Create Search Resource and utilize Retrieval Augmented Generation (RAG) to generate response

In this notebook, we will be generate 

Here are the steps we will execute:
- Cleanse the data within the CSV file
- Upload file to Blob resource
- Create Search Index and Associate Indexer to ingest data
- Now run the indexer to ingest embedded data into AI Search 
- Next step would be querying the search for question/answer 

In [1]:
%pip install numpy pandas

   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/12.7 MB 9.7 MB/s eta 0:00:02
   -------------- ------------------------- 4.5/12.7 MB 13.8 MB/s eta 0:00:01
   -------------- ------------------------- 4.7/12.7 MB 8.3 MB/s eta 0:00:01
   ------------------ --------------------- 5.8/12.7 MB 7.1 MB/s eta 0:00:01
   ---------------------- ----------------- 7.1/12.7 MB 7.5 MB/s eta 0:00:01
   -------------------------------- ------- 10.2/12.7 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------  12.6/12.7 MB 8.9 MB/s eta 0:00:01
   ---------------------------------------- 12.7/12.7 MB 8.5 MB/s  0:00:01
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ------------- -------------------------- 3.4/9.8 MB 18.0 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.8 MB 15.5 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.8 MB 14.4 MB/s eta 0:00:01
   ----------

In [168]:
import pandas as pd
import numpy as np
import re
from IPython.display import display 

# Load the dataset
df = pd.read_csv("data\\cloudnova_invoices.csv")

# Trim Whitespacesin Column Names to standardize them
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

# Normailize Churned data into 0/1
df["churned"] = df["churned"].astype(str).str.lower().map({
    "yes": 1, "y": 1, "true": 1, "1": 1,
    "no": 0, "n": 0, "false": 0, "0": 0
})

# Standardize billing cycle
billing_map = {
    "monthly": "monthly",
    "month": "monthly",
    "annual": "annual",
    "yearly": "annual",
    "annually": "annual"
}
df["billing_cycle"] = df["billing_cycle"].str.lower().map(billing_map)


plan_map = {
    "starter": "starter", "start": "starter",
    "pro": "pro", "professional": "pro",
    "enterprise": "enterprise", "ent": "enterprise"
}
df["plan"] = df["plan"].str.lower().map(plan_map)

# Normalize status values
status_map = {
    "paid": "paid", "complete": "paid", "completed": "paid",
    "pending": "pending", "awaiting": "pending",
    "failed": "failed", "error": "failed", "declined": "failed",
    "refunded": "refunded", "charged_back": "refunded",
    "canceled": "canceled", "cancelled": "canceled",
    "void": "void"
}
df["status"] = df["status"].str.lower().map(status_map)



# Fix email _at_ by replacing with @
df["contact_email"] = df["contact_email"].str.replace("_at_", "@", regex=False)

# Associate the Symbol with the corresponding currency code
def extract_currency(val):
    if isinstance(val, str):
        if "usd" in val.lower(): return "USD"
        if "eur" in val.lower(): return "EUR"
        if "gbp" in val.lower(): return "GBP"
        if "$" in val: return "USD"
        if "€" in val: return "EUR"
        if "£" in val: return "GBP"
    return np.nan

df["currency_extracted"] = df["amount"].apply(extract_currency)

# Clean numeric amount values
def clean_amount(val):
    if isinstance(val, str):
        val = re.sub(r"[^\d\.-]", "", val)  # remove currency symbols & commas
    try:
        return float(val)
    except:
        return np.nan

df["amount_clean"] = df["amount"].apply(clean_amount)

# Flag the invalid signup dates
df["invalid_invoice_date"] = df["invoice_date"] < df["signup_date"]

# Find refund mismatches: refunded status but positive amount
df["refund_mismatch"] = (df["status"] == "refunded") & (df["amount_clean"] > 0)

rates = {"USD": 1, "EUR": 1.08, "GBP": 1.27}

df["amount_usd"] = df.apply(
    lambda row: row["amount_clean"] * rates.get(row["currency_extracted"], 1),
    axis=1
)

# for annual subscription, subtract discount from amount_usd to get the monthly equivalent
def adjust_annual_discount(row):

    if row["billing_cycle"] == "annual" and not pd.isna(row["amount_usd"]):
        return row["amount_usd"] * (1 - row["discount_pct"]) 
    return row["amount_usd"]

df["amount_usd"] = df.apply(adjust_annual_discount, axis=1)


# Set amount_usd to negative if not already negative for refunded invoices
df.loc[(df["status"] == "refunded") & (df["amount_usd"] > 0), "amount_usd"] *= -1

# Create a new dataframe and drop all the columns that are not paid or refunded invoices
df = df[df["status"].isin(["paid", "refunded"])]

df["invoice_date"] = pd.to_datetime(df["invoice_date"], format="mixed", errors="coerce").dt.strftime("%Y-%m-%d")

# Find number of duplicated rows based on account_id, invoice_date, and amount_clean and 
df["duplicate"] = df.duplicated(subset=["account_id", "invoice_date", "amount_clean"], keep=False)  

# remove duplicates based on account_id, invoice_date, and amount_clean, keeping the first occurrence
df = df.drop_duplicates(subset=["account_id", "invoice_date", "amount_clean"], keep="first")

# Compute MRR
def compute_mrr(row):
    amount = row["amount_clean"]
    seats = row["seats"]
    status = row["status"]
    billing_cycle = row["billing_cycle"]
    mrr = 0

    if status != "paid" or pd.isna(amount):
        return 0
    
    if billing_cycle == "annual" :
       base_mrr = amount / 12
       return  base_mrr
    else:
        return amount

# Plan is empty in some cases hence cannot be considered in MRR calculations
df["mrr_usd"] = df.apply(compute_mrr, axis=1)




# drop columns discount_pct, payment_method, signup_date, amount, currency_extracted, currency, invalid_invoice_date, refund_mismatch, duplicate, plan, amount_clean, support_tickets, billing_cycle, and seats
df = df.drop(columns=["discount_pct", "payment_method", "signup_date", "amount", "currency_extracted", "currency", "invalid_invoice_date", "refund_mismatch", "plan", "amount_clean", "support_tickets", "billing_cycle", "seats"])

# rename amount_usd to amount and mrr_usd to mrr
df = df.rename(columns={"amount_usd": "amount", "mrr_usd": "mrr"})

# add a key field that stored the base64 encoded value of the invoice_id and ensure it is key field in the indexer and the first column
df["key"] = df["invoice_id"].apply(lambda x: x.encode("utf-8").hex())
df = df[['key'] + [col for col in df.columns if col != 'key']]

df.to_csv('data\\cleaned_invoices.csv', index=False)
display(df.head(50))


,key,invoice_id,account_id,account_name,contact_email,region,industry,status,invoice_date,churned,csat_score,amount,duplicate,mrr
1,494e562d313032373436,INV-102746,ACC-7599,Wayne Co,billing@wayne.com,LATAM,Education,paid,2026-05-20,0,4.0,4.900000e+03,False,408.333333
3,494e562d313031333535,INV-101355,ACC-8014,Oscorp Holdings,NaN,NaN,Retail,refunded,2024-02-13,0,4.0,-4.900000e+03,False,0.000000
4,494e562d313034393530,INV-104950,ACC-8471,Prestige Co,billing@prestige.com,NaN,Energy,paid,2022-02-03,0,NaN,1.225000e+04,False,1020.833333
5,494e562d313033323634,INV-103264,ACC-3565,Cyberdyne Group,billing@cyberdyne.com,NaN,Manufacturing,paid,2025-11-22,1,5.0,-2.364740e+05,False,1037.166667
6,494e562d313030383633,INV-100863,ACC-4859,Nakatomi Corp,billing@nakatomi.com,NaN,Education,paid,2024-04-30,1,5.0,2.450000e+02,False,245.000000
7,494e562d313032353737,INV-102577,ACC-7705,Massive Dynamic Corp,billing@massive.com,EMEA,Energy,paid,2023-09-04,0,5.0,-5.831000e+04,False,347.083333
8,494e562d313033343936,INV-103496,ACC-2159,Vandelay Ltd,billing@vandelay.com,EMEA,Logistics,paid,2023-01-03,0,4.0,1.225000e+03,False,1225.000000
9,494e562d313030313930,INV-100190,ACC-4351,Wonka Group,billing@wonka.com,NaN,Gaming,paid,2025-02-25,0,5.0,5.292000e+04,False,4410.000000
10,494e562d313033313133,INV-103113,ACC-2800,Wayne Labs,billing@wayne.com,EMEA,Travel,paid,2026-04-13,0,5.0,2.990000e+05,False,24916.666667
11,494e562d313034353234,INV-104524,ACC-9876,Prestige LLC,billing@prestige.com,LATAM,Retail,paid,2026-12-16,0,4.0,1.257300e+04,False,1047.750000



ERROR: ERROR: {"code": "InvalidTemplate", "message": "Deployment template validation failed: 'The template resource 'https://foundry-udemy-mg.cognitiveservices.azure.com/' of type 'Microsoft.Search/searchServices' at line '129' and column '53' has an invalid name. The name 'https://foundry-udemy-mg.cognitiveservices.azure.com/' contains '2' name segment(s) separated by '/', but '1' segment(s) were expected. Please see https://aka.ms/arm-syntax-resources for usage details.'.", "additionalInfo": [{"type": "TemplateViolation", "info": {"lineNumber": 129, "linePosition": 53, "path": "properties.template.resources[0].type"}}]}



In [4]:
from azure.identity import DefaultAzureCredential

# Single credential object reused across all SDK clients
credential = DefaultAzureCredential()
print("Credential initialized — using DefaultAzureCredential")

Credential initialized — using DefaultAzureCredential


In [183]:
# Create a blob container inside the storage account.
# The signed-in user also needs a data-plane role before blob uploads work.
import subprocess
import shutil
import time
from azure.storage.blob import BlobServiceClient

storage_account_name = "myfoundryudemystorage"

blob_service_client = BlobServiceClient(
    account_url=f"https://{storage_account_name}.blob.core.windows.net",
    credential=credential,
)
BLOB_CONTAINER_NAME= "cloudnova"
container_client = blob_service_client.get_container_client(BLOB_CONTAINER_NAME)
if container_client.exists():
    print(f"Blob container already exists: {BLOB_CONTAINER_NAME}")
else:
    container_client.create_container()
    print(f"Blob container created: {BLOB_CONTAINER_NAME}")


Blob container already exists: cloudnova


In [7]:
%pip install azure-search-documents==11.7.0b2 openai azure-identity python-dotenv

  Attempting uninstall: azure-search-documents
    Found existing installation: azure-search-documents 12.0.0
    Uninstalling azure-search-documents-12.0.0:
      Successfully uninstalled azure-search-documents-12.0.0
Note: you may need to restart the kernel to use updated packages.


In [184]:
# Upload sample documents to the blob container
from pathlib import Path

container_client = blob_service_client.get_container_client(BLOB_CONTAINER_NAME)

sample_dir = Path("data")
if not sample_dir.exists():
    repo_relative = Path("Cloud-Nova/data")
    if repo_relative.exists():
        sample_dir = repo_relative
    else:
        raise FileNotFoundError(
            "Sample data directory not found. Please ensure the 'data' directory exists in the current working directory or in the 'Cloud-Nova' subdirectory."
        )

cloudnova_files = ["cleaned_invoices.csv"]

for filename in cloudnova_files:
    filepath = sample_dir / filename
    with filepath.open("rb") as f:
        container_client.upload_blob(name=filename, data=f, overwrite=True)
    print(f"Uploaded: {filename}")

print(f"\nAll {len(cloudnova_files)} documents uploaded to '{BLOB_CONTAINER_NAME}' container.")

Uploaded: cleaned_invoices.csv

All 1 documents uploaded to 'cloudnova' container.


In [185]:
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SearchIndexerDataSourceConnection,
    SearchIndexerDataContainer,
    SearchIndexer,
    FieldMapping,
    FieldMappingFunction,
    SearchableField,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SimpleField,
)

search_endpoint = "https://myfoundryudemysearch.search.windows.net"
index_name = "cloudnova-invoices-index"
index = SearchIndex(
    name=index_name,
    fields=[
       SearchField(
                 name="key",           # base64-encoded invoice_id
                 type=SearchFieldDataType.String,
                 key=True,
                 filterable=True,
             ),

        SimpleField(
            name="invoice_id",        # e.g. "INV-102746"
            type=SearchFieldDataType.String,
            filterable=True,
        ),
        # ── Account identifiers ────────────────────────────────────────────────
        SimpleField(
            name="account_id",           # e.g. "ACC-7599"
            type=SearchFieldDataType.String,
            filterable=True,
        ),
        SearchableField(
            name="account_name",         # e.g. "Wayne Co"
            type=SearchFieldDataType.String,
            filterable=True,
            facetable=True,
            sortable=True,
            analyzer_name="en.microsoft",
        ),
        SearchableField(
            name="contact_email",        # e.g. "billing@wayne.com" — nullable
            type=SearchFieldDataType.String,
            filterable=True,
            analyzer_name="keyword",     # exact-match email lookups
        ),

        # ── Categorical / segmentation fields ──────────────────────────────────
        SimpleField(
            name="region",               # LATAM | EMEA | APAC | (empty)
            type=SearchFieldDataType.String,
            filterable=True,
            facetable=True,
        ),
        SimpleField(
            name="industry",             # Education | Retail | Energy …
            type=SearchFieldDataType.String,
            filterable=True,
            facetable=True,
        ),
        SimpleField(
            name="status",               # paid | refunded
            type=SearchFieldDataType.String,
            filterable=True,
            facetable=True,
        ),

        # ── Temporal ───────────────────────────────────────────────────────────
        SimpleField(
            name="invoice_date",         # e.g. "2026-05-20"
            type=SearchFieldDataType.DateTimeOffset,
            filterable=True,
            sortable=True,
        ),

        # ── Flags & scores ─────────────────────────────────────────────────────
        SimpleField(
            name="churned",              # 0 or 1
            type=SearchFieldDataType.Int32,
            filterable=True,
            facetable=True,
        ),
        SimpleField(
            name="csat_score",           # 1.0 – 5.0, nullable
            type=SearchFieldDataType.Double,
            filterable=True,
            sortable=True,
        ),

        # ── Financial metrics ──────────────────────────────────────────────────
        SimpleField(
            name="amount",               # invoice total; can be negative (refund)
            type=SearchFieldDataType.Double,
            filterable=True,
            sortable=True,
        ),
        SimpleField(
            name="mrr",                  # monthly recurring revenue contribution
            type=SearchFieldDataType.Double,
            filterable=True,
            sortable=True,
        ),
    ],

    # ── Semantic configuration (required for Foundry IQ / agentic retrieval) ──
    semantic_search=SemanticSearch(
        configurations=[
            SemanticConfiguration(
                name="default",
                prioritized_fields=SemanticPrioritizedFields(
                    # Best proxy for a "title" in an invoice dataset
                    title_field=SemanticField(field_name="account_name"),
                    # Searchable text fields ranked for semantic relevance
                    content_fields=[
                        SemanticField(field_name="contact_email"),
                    ],
                    # High-signal categorical fields used as keyword context
                    keywords_fields=[
                        SemanticField(field_name="industry"),
                        SemanticField(field_name="region"),
                        SemanticField(field_name="status"),
                    ],
                ),
            )
        ]
    ),
)


# --- 1. Create the search index ---
index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)
index_client.create_or_update_index(index)
print(f"Index ready: {index_name}")

Index ready: cloudnova-invoices-index


In [186]:
# --- 2. Create the data source connection ---
from dotenv import load_dotenv

indexer_client = SearchIndexerClient(endpoint=search_endpoint, credential=credential)


subscription_id = "98d43133-a36c-4ce0-9f59-d13015b39c19"
resource_group = "foundryudemy-rg"     
storage_account_name = "myfoundryudemystorage"      

connection_string = (
    f"ResourceId=/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}"
    f"/providers/Microsoft.Storage/storageAccounts/{storage_account_name};"
)

data_source = SearchIndexerDataSourceConnection(
    name="cloudnova-invoices-datasource",
    type="azureblob",
    connection_string=connection_string,
    container=SearchIndexerDataContainer(name=BLOB_CONTAINER_NAME)
)

indexer_client.create_or_update_data_source_connection(data_source)
print(f"Data source ready: {data_source.name}")


Data source ready: cloudnova-invoices-datasource


In [202]:
from azure.core.exceptions import ResourceNotFoundError
from azure.search.documents.indexes import SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndexer,
    IndexingParameters,
    IndexingParametersConfiguration,
    FieldMapping,
)

indexer_client = SearchIndexerClient(
    endpoint=search_endpoint,
    credential=credential
)
# Step 1 — Hard delete to wipe all cached state
print("Deleted.")

# Step 2 — Recreate with parsingMode explicitly set
indexer = SearchIndexer(
    name="cloudnova-invoices-indexer",
    data_source_name="cloudnova-invoices-datasource",
    target_index_name="cloudnova-invoices-index",

   # Generate field mappings to ensure the indexer correctly maps CSV columns to index fields
    field_mappings=[
        FieldMapping(
                    source_field_name="metadata_storage_path",
                    target_field_name="key",
                    mapping_function=FieldMappingFunction(name="base64Encode")
                ),  
        FieldMapping(source_field_name="invoice_id",    target_field_name="invoice_id"),
        FieldMapping(source_field_name="account_id",    target_field_name="account_id"),
        FieldMapping(source_field_name="account_name",  target_field_name="account_name"),
        FieldMapping(source_field_name="contact_email", target_field_name="contact_email"),
        FieldMapping(source_field_name="region",        target_field_name="region"),
        FieldMapping(source_field_name="industry",      target_field_name="industry"),
        FieldMapping(source_field_name="status",        target_field_name="status"),
        FieldMapping(source_field_name="invoice_date",  target_field_name="invoice_date"),
        FieldMapping(source_field_name="churned",       target_field_name="churned"),
        FieldMapping(source_field_name="csat_score",    target_field_name="csat_score"),
        FieldMapping(source_field_name="amount",        target_field_name="amount"),
        FieldMapping(source_field_name="mrr",           target_field_name="mrr")
    ]
)

indexer_client.create_indexer(indexer)
print("Indexer recreated.")


Deleted.
Indexer recreated.


In [200]:
import time
from azure.core.exceptions import HttpResponseError

# Trigger the indexer to start processing documents.
try:
    indexer_client.run_indexer("cloudnova-invoices-indexer")
    print("Indexer started. Waiting for completion...")
except HttpResponseError as ex:
    if "already running" in str(ex).lower():
        print("Indexer is already running. Waiting for the current run to finish...")
    else:
        raise

# Poll until the indexer finishes
while True:
    status = indexer_client.get_indexer_status("cloudnova-invoices-indexer")
    last_result = status.last_result

    if last_result and last_result.status in (
        "success",
        "transientFailure",
        "persistentFailure",
    ):
        print(f"\nIndexer finished.")
        print(f"  Status:              {last_result.status}")
        print(f"  Documents processed: {last_result.item_count}")
        print(f"  Documents failed:    {last_result.failed_item_count}")
        if last_result.errors:
            for error in last_result.errors:
                print(f"  Error: {error.error_message}")
        if last_result.status == "persistentFailure" or (last_result.failed_item_count or 0) > 0:
            raise RuntimeError("Indexer completed with failures. Review the errors above before continuing.")
        break

    print("  Indexing in progress...")

Indexer started. Waiting for completion...

Indexer finished.
  Status:              transientFailure
  Documents processed: 1
  Documents failed:    1
  Error: Could not parse document. Invalid document key: 'https://myfoundryudemystorage.blob.core.windows.net/cloudnova/cleaned_invoices.csv'. Keys can only contain letters, digits, underscore (_), dash (-), or equal sign (=). Please see https://docs.microsoft.com/azure/search/search-howto-indexing-azure-blob-storage#DocumentKeys


RuntimeError: Indexer completed with failures. Review the errors above before continuing.